# Under-ice Argo trajectory reconstruction


## 1. Parameters


In [ ]:
from pathlib import Path

# ============================================================
# PARAMETERS
# ============================================================

# Float and input files
float_name = "6903562"
file_bathy = "../0_data/bathy_isas17.nc"
file_prof = f"../0_data/Argo/{float_name}/{float_name}_prof.nc"
file_Rtraj = f"../0_data/Argo/{float_name}/{float_name}_Rtraj.nc"
file_meta = f"../0_data/Argo/{float_name}/{float_name}_meta.nc"

# GLORYS12 daily current data directory
base_dir = Path("/home/ref-ocean-reanalysis/global-reanalysis-phy-001-030-daily")

# Smoothing methods to run
use_enks = True
use_ffbs = True
if not (use_enks or use_ffbs):
    raise ValueError("Select at least one method: use_enks=True and/or use_ffbs=True.")

# Ensemble / particle sizes
Ne_enkf = 1000
Ne_ffbs = 5000

# Uncertainties
std_current = 0.03   # m/s: standard deviation of current uncertainty
std_gps = 10         # m
std_pres = 200       # m

# Additional observation constraints
# The land constraint is always active.
use_pv = False
use_speed = False

# Output and plotting
output_dir = Path("Exports")
figure_dir = output_dir / "Figures"
csv_dir = output_dir / "Trajectories"
figure_dir.mkdir(parents=True, exist_ok=True)
csv_dir.mkdir(parents=True, exist_ok=True)

chi2_95 = 5.991
ellipse_step = 1
plot_start_cycle = 0
plot_end_cycle = None

# Plot colors
color_recorded = "limegreen"
color_enks = "purple"
color_ffbs = "orange"
color_gnss = "black"

# Figure sizes
figsize_map = (9, 8)
figsize_uncertainty = (11, 5)
figsize_diagnostics = (14, 12)

print(f"Float: {float_name}")
print(f"EnKS: {'ON' if use_enks else 'OFF'}")
print(f"FFBS: {'ON' if use_ffbs else 'OFF'}")
print(f"Current data directory: {base_dir}")
print("Land constraint: ON")
print("Numerical integration: RK45")
print("EnKF inflation: OFF")
print("Particle roughening: OFF")


## 2. Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

from pathlib import Path
from pyproj import Transformer
from scipy.integrate import solve_ivp
from scipy.interpolate import RegularGridInterpolator

import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.patches import Polygon


In [ ]:
### Coordinate transformation 
transformer = Transformer.from_crs("EPSG:4326",    # coordonnées géographiques (lat/lon WGS84)
                                   "EPSG:3995",    # projection arctique stéréographique
                                   always_xy=True) # important : (lon, lat) et non (lat, lon)

def xy2lonlat (x,y) :
    lon, lat = transformer.transform(x, y, direction="INVERSE")
    return lon, lat

def lonlat2xy (lon, lat):
    x,y = transformer.transform(lon, lat)
    return x,y

In [ ]:
### Bathymetry 
bathy_ds = xr.open_dataset(file_bathy)
def bathy(lon,lat):
    z = bathy_ds.bathymetry.sel(latitude=lat, longitude=lon, method="nearest").values
    return z

def bathy_arr(lon, lat):
    lon = xr.DataArray(lon, dims="points")
    lat = xr.DataArray(lat, dims="points")
    z = bathy_ds.bathymetry.sel(longitude=lon, latitude=lat,  method="nearest").values
    return z

def bathy_xy(x,y):
    lon, lat = xy2lonlat(x,y)
    z = bathy(lon, lat)
    return z

In [ ]:
### Potential vorticity

Omega = 7.2921e-5  # rad/s, vitesse angulaire terrestre

def f_coriolis(lat):
    return 2 * Omega * np.sin(np.radians(lat))

def pv_xy(x, y):
    lon, lat = xy2lonlat(x, y)
    h = bathy(lon, lat)
    if h <= 0:
        return np.nan 
    return f_coriolis(lat) / h

def pv(lon, lat):
    h = bathy_arr(lon, lat)      # h est de taille (N,)
    h = np.where(h <= 0, np.nan, h)
    return f_coriolis(lat) / h

In [ ]:
### Float speed

def speed_arr(xs, ys, times):
    n = len(xs)
    speed = np.full(n, np.nan)

    dt_sec = np.diff(times.values).astype('timedelta64[s]').astype(float)
    dx = np.diff(xs)
    dy = np.diff(ys)
    dist = np.sqrt(dx**2 + dy**2)

    with np.errstate(invalid='ignore', divide='ignore'):
        v = np.where(dt_sec > 0, dist / dt_sec, np.nan)

    speed[1:] = v
    return speed


def speed_xy(x, y, x_past, y_past, t, t_past):
    
    dt_sec = t - t_past
    dx = x - x_past
    dy =  y - y_past
    dist = np.sqrt(dx**2 + dy**2)

    with np.errstate(invalid='ignore', divide='ignore'):
        v = np.where(dt_sec > 0, dist / dt_sec, np.nan)

    return v

In [ ]:
### Float data
prof_ds = xr.open_dataset(file_prof)
float_ds = xr.open_dataset(file_Rtraj)
meta_ds = xr.open_dataset(file_meta)

cycle_time = prof_ds.JULD.values
cycle_profil = prof_ds.CYCLE_NUMBER.values
cycle_lat = prof_ds.LATITUDE.values 
cycle_lon = prof_ds.LONGITUDE.values
cycle_qc = prof_ds.POSITION_QC.values
cycle_mission = prof_ds.CONFIG_MISSION_NUMBER.values
profil_grounded = float_ds.GROUNDED.values
pres = prof_ds.PRES.values
mission_number = meta_ds.CONFIG_MISSION_NUMBER.values

n_cycle = np.shape(cycle_time)[0]
cycle_x, cycle_y = lonlat2xy(cycle_lon, cycle_lat)

# Keep only profiles with valid GNSS positioning for the observation vector
valid = (cycle_qc == b'1')
print(f"{np.sum(valid)} valid GNSS positions")
print(f"{np.sum(~valid)} invalid GNSS positions")
invalid_idx = np.where(~valid)[0]
if len(invalid_idx) > 0:
    print(f"Cycles with invalid GNSS position: {invalid_idx}")
    
lats = cycle_lat.copy()
lons = cycle_lon.copy()
lats[~valid] = np.nan
lons[~valid] = np.nan


# Convert timestamps
times = pd.to_datetime(cycle_time)


# Grounding pressure
grounded_pres = np.full_like(cycle_time, np.nan, dtype=float)
for c in range (n_cycle) :
    ip = int(cycle_profil[c]) -1
    if ip >= len(profil_grounded) : 
        continue
    
    if profil_grounded[ip] == b'Y':
        if np.all(np.isnan(pres[c])):
            print(f"Warning: pres[{c}] contains only NaN values")
        else:
            grounded_pres[c] = np.nanmax(pres[c])
print(f"{np.sum(np.isfinite(grounded_pres))} grounded observations available")


# Parking depth
param_names = meta_ds.CONFIG_PARAMETER_NAME.values
idx_parkpres = np.where([b"CONFIG_ParkPressure_dbar" in s for s in param_names])[0][0]
mission_parkpres = meta_ds.CONFIG_PARAMETER_VALUE.values[:, idx_parkpres]

parking_depth = np.full_like(cycle_time, np.nan, dtype=float)
for c in range(n_cycle):
    im = int(cycle_mission[c]) -1
    parking_depth[c] = mission_parkpres[im]
parking_depth = np.nanmin((parking_depth, grounded_pres), axis = 0)

# Profile depth    
idx_profpres = np.where([b"CONFIG_ProfilePressure_dbar" in s for s in param_names])[0][0]
mission_profpres = meta_ds.CONFIG_PARAMETER_VALUE.values[:, idx_profpres]    
profil_depth = np.full_like(cycle_time, np.nan, dtype=float)
for c in range(n_cycle):
    im = int(cycle_mission[c]) -1
    profil_depth[c] = mission_profpres[im]

tol_profil = 100  # m, tolerance for considering the profile complete
reached_profil = np.full(n_cycle, False)
for c in range(n_cycle):
    if np.isfinite(grounded_pres[c]):
        continue 
    if np.all(np.isnan(pres[c])):
        continue
    if np.abs(np.nanmax(pres[c]) - profil_depth[c]) < tol_profil:
        reached_profil[c] = True
    
# Projected coordinates
xs, ys = lonlat2xy(lons, lats)

# Observation vector
Y = np.zeros((3,times.shape[0]))
Y[0,:] = xs
Y[1,:] = ys
Y[2,:] = grounded_pres




mask_valid = np.isfinite(times)
valid = valid[mask_valid]

# Apply the valid-time mask to all cycle-dependent arrays
cycle_time      = cycle_time[mask_valid]
cycle_profil    = cycle_profil[mask_valid]
cycle_lat       = cycle_lat[mask_valid]
cycle_x         = cycle_x[mask_valid]
cycle_y         = cycle_y[mask_valid]
cycle_lon       = cycle_lon[mask_valid]
cycle_qc        = cycle_qc[mask_valid]
cycle_mission   = cycle_mission[mask_valid]
pres            = pres[mask_valid]

lats            = lats[mask_valid]
lons            = lons[mask_valid]
times           = times[mask_valid]

grounded_pres   = grounded_pres[mask_valid]
parking_depth   = parking_depth[mask_valid]
profil_depth    = profil_depth[mask_valid]
reached_profil  = reached_profil[mask_valid]

xs              = xs[mask_valid]
ys              = ys[mask_valid]

# Update the number of cycles and observation vector Y
n_cycle = np.shape(cycle_time)[0]

Y = np.zeros((3, n_cycle))
Y[0, :] = xs
Y[1, :] = ys
Y[2, :] = grounded_pres


## 3. Float diagnostics

This figure summarizes the float observations used by the reconstruction and explicitly shows GNSS validity.

In [ ]:
# Compute diagnostics along the recorded trajectory.
bathy_recorded = bathy_arr(cycle_lon, cycle_lat)
pv_recorded = pv(cycle_lon, cycle_lat)
speed_recorded = speed_arr(cycle_x, cycle_y, times)
gnss_valid = valid.copy()

fig, axes = plt.subplots(4, 1, figsize=figsize_diagnostics, sharex=True)

# Measured pressure, parking depth, and grounding information.
ax = axes[0]
pressure_max = np.nanmax(pres, axis=1)
ax.plot(times, pressure_max, label="Maximum measured pressure", linewidth=1.5)
ax.plot(times, parking_depth, label="Parking depth", linewidth=2)
ax.plot(times, grounded_pres, label="Grounded pressure", linewidth=2, linestyle="--")
ax.set_ylabel("Pressure / depth [dbar / m]")
ax.invert_yaxis()
ax.grid(alpha=0.3)
ax.legend(loc="best")
ax.set_title(f"Float observations and diagnostics — WMO {float_name}")

# Bathymetry, parking depth, and grounding information.
ax = axes[1]
ax.plot(times, bathy_recorded, label="Bathymetry", linewidth=2)
ax.plot(times, parking_depth, label="Parking depth", linewidth=2)
ax.plot(times, grounded_pres, label="Grounded pressure", linewidth=1.5, linestyle="--")
ax.set_ylabel("Depth [m]")
ax.invert_yaxis()
ax.grid(alpha=0.3)
ax.legend(loc="best")

# Potential vorticity and float speed.
ax = axes[2]
ax.plot(times, pv_recorded, label="Potential vorticity", linewidth=1.8)
ax.set_ylabel("PV [s$^{-1}$ m$^{-1}$]")
ax.grid(alpha=0.3)
ax.legend(loc="upper left")
ax2 = ax.twinx()
ax2.plot(times, speed_recorded, label="Float speed", linewidth=1.8, linestyle="--")
ax2.set_ylabel("Speed [m/s]")
ax2.legend(loc="upper right")

# GNSS validity.
ax = axes[3]
ax.step(times, gnss_valid.astype(int), where="mid", linewidth=2, label="Valid GNSS position")
ax.scatter(times[gnss_valid], np.ones(np.sum(gnss_valid)), s=18, label="Valid GNSS")
ax.scatter(times[~gnss_valid], np.zeros(np.sum(~gnss_valid)), s=18, label="Invalid / missing GNSS")
ax.set_yticks([0, 1])
ax.set_yticklabels(["Invalid / missing", "Valid"])
ax.set_ylim(-0.15, 1.15)
ax.set_ylabel("GNSS")
ax.set_xlabel("Date")
ax.grid(alpha=0.3)
ax.legend(loc="best")

fig.tight_layout()
fig.savefig(figure_dir / f"float_diagnostics_{float_name}.png", dpi=300, bbox_inches="tight")
plt.show()


## 4. Current data and model

In [ ]:
### Load current data
year_min, year_max = np.nanmin(times.year), np.nanmax(times.year)
lon_min, lon_max = np.nanmin(lons) - 5, np.nanmax(lons) + 5
lat_min, lat_max = np.nanmin(lats) - 5, np.nanmax(lats) + 5
z_min, z_max = np.nanmin(parking_depth) - 100, np.nanmax(parking_depth) + 100

map_extent = [lon_min, lon_max, lat_min, lat_max]


files = []
for year in range(int(year_min), int(year_max) + 1):
    files.extend(sorted((base_dir / str(year)).glob("*/mercatorglorys12v1_gl12_mean_*.nc")))

print(f"{len(files)} current files found")

def preprocess(ds):
    return (ds[["uo", "vo"]].sel(longitude=slice(lon_min, lon_max), latitude=slice(lat_min, lat_max), depth=slice(z_min, z_max)))

ds= xr.open_mfdataset(files, combine="by_coords", preprocess=preprocess, parallel=True) # chunks={"time": 30, "latitude": 200, "longitude": 200,},)

ds_g12 = ds.load() 


In [ ]:
### Ocean currents
u_interp = RegularGridInterpolator((ds_g12.time.values.astype('datetime64[s]').astype(float), ds_g12.depth.values, ds_g12.latitude.values, ds_g12.longitude.values), ds_g12.uo.values, bounds_error=False, fill_value=0.0)
v_interp = RegularGridInterpolator((ds_g12.time.values.astype('datetime64[s]').astype(float), ds_g12.depth.values, ds_g12.latitude.values, ds_g12.longitude.values), ds_g12.vo.values, bounds_error=False, fill_value=0.0)

def current_xy(x, y, time, z):
    lon, lat = xy2lonlat(x, y)
    t = np.datetime64(time).astype('datetime64[s]').astype(float)
    u = float(u_interp((t, z, lat, lon)))
    v = float(v_interp((t, z, lat, lon)))
    if not np.isfinite(u):
        u = 0.
    if not np.isfinite(v):
        v = 0.
    return u, v


def current_xy_batch(x_arr, y_arr, time, z):
    lon, lat = xy2lonlat(x_arr, y_arr)
    t = np.datetime64(time).astype('datetime64[s]').astype(float)
    N = len(x_arr)
    pts = np.column_stack([np.full(N, t), np.full(N, z), lat, lon])
    u = u_interp(pts)
    v = v_interp(pts) 
    return np.nan_to_num(u), np.nan_to_num(v)



## 5. Dynamic model and observation space

In [ ]:
def m_batch(X_past, t_past, t, time, time_past, z):
    N = X_past.shape[1]
    dt = t - t_past

    def derive(tau, state):
        x = state[:N]
        y = state[N:]
        # Temporal interpolation between time_past and time according to tau
        if dt != 0 :
            frac = (tau - t_past) / dt  
        else :
            frac = 0.0
        time_interp = time_past + (time - time_past) * frac
        u, v = current_xy_batch(x, y, time_interp, z)
        return np.concatenate([u, v])

    y0 = np.concatenate([X_past[0], X_past[1]])  # (2 * Ne,)

    sol = solve_ivp(derive, (t_past, t), y0, method="RK45", max_step=dt)

    if not sol.success:
        print("RK45 integration failed:", sol.message)
        return np.full_like(X_past, np.nan)

    y_final = sol.y[:, -1]
    x_new = y_final[:N]
    y_new = y_final[N:]
    return np.array([x_new, y_new])

In [ ]:
### Observation model 
   
def h(X, pv_ref=None, X_past=None, t_k=None, t_km1=None):
    x, y = X
    z_bathy = bathy_xy(x, y)
    h_x = [x, y, z_bathy]

    if use_pv :
        pv = pv_xy(x, y)
        h_x.append(pv - pv_ref)

    if use_speed :
        x_past, y_past = X_past
        s = speed_xy(x, y, x_past, y_past, t_k, t_km1)
        h_x.append(s)

    return h_x



R_diag = [std_gps**2, std_gps**2, std_pres**2]
p = 3
y = Y.copy()

if use_pv :
    R_diag.append(std_pv**2)
    p+=1
    y = np.vstack((y, np.zeros((1,nb))))
    
if use_speed :
    R_diag.append(std_speed**2)
    p+=1
    y = np.vstack((y, speed_target*np.ones((1,nb))))


R = np.diag(R_diag)


In [ ]:
### Filter initialization

n = 2
nb = len(times)
x_0, y_0 = cycle_x[0], cycle_y[0]
X_0 = np.array([x_0, y_0])
P_0 = np.diag([std_gps**2, std_gps**2])
sigma0 = np.sqrt(np.diag(P_0))
t_eval = (times - times[0]).total_seconds()
t_eval = np.array(t_eval)

## 6. EnKF and EnKS

In [ ]:
if use_enks:
    ### Ensemble Kalman filter

    # Ensemble Kalman initialization
    x_f_enkf = np.zeros((n, nb))
    P_f_enkf = np.zeros((n, n, nb))
    x_a_enkf = np.zeros((n, nb))
    P_a_enkf = np.zeros((n, n, nb))

    x_a_enkf_tmp = np.zeros((n, Ne_enkf))
    x_f_enkf_tmp = np.zeros((n, Ne_enkf))
    y_f_enkf_tmp = np.zeros((p, Ne_enkf))

    x_a_enkf_full = np.zeros((n, Ne_enkf, nb))
    x_f_enkf_full = np.zeros((n, Ne_enkf, nb))

    if use_pv : 
        pv0 = pv_xy(X_0[0], X_0[1])
     
        pv_a = np.zeros((nb))
        pv_f = np.zeros((nb))
        pv_s = np.zeros((nb))

        pv_f[0] = pv0
        pv_a[0] = pv0

    if use_speed : 
        speed_a_enkf = np.zeros((nb))
        speed_a_enkf[0] = np.nan
    




    # initial step
    for i in range(Ne_enkf):
        x_a_enkf_tmp[:, i] = np.random.multivariate_normal(X_0, P_0)

    x_a_enkf[:, 0] = np.mean(x_a_enkf_tmp, axis=1) # initial state
    P_a_enkf[:, :, 0] = np.cov(x_a_enkf_tmp) # initial state covariance
    x_a_enkf_full[:,:,0] = x_a_enkf_tmp

    x_f_enkf_tmp=x_a_enkf_tmp.copy()
    x_f_enkf[:, 0] = x_a_enkf[:, 0]
    P_f_enkf[:, :, 0] = P_a_enkf[:, :, 0]
    x_f_enkf_full[:,:,0] = x_f_enkf_tmp





    # forward in time
    for k in range(1, nb):
        if k%10 == 0 : 
            print(f"{k}/{nb}")

        dt = t_eval[k] - t_eval[k-1]
        time = times[k]
        Q = np.diag([(std_current*dt)**2, (std_current*dt)**2])
        z = parking_depth[k]
    
        Q_batch = np.random.multivariate_normal(np.zeros(n), Q, size=Ne_enkf).T  # (n, Ne)
        x_f_enkf_tmp = m_batch(x_a_enkf_tmp, t_eval[k-1], t_eval[k], times[k], times[k-1], z)  + Q_batch
        for i in range(Ne_enkf): # prediction step
            y_f_enkf_tmp[:, i] = h(x_f_enkf_tmp[:, i], pv_a[k-1]  if use_pv else None, x_a_enkf_tmp[:,i] if use_speed else None, t_eval[k], t_eval[k-1])  + np.random.multivariate_normal(np.zeros(p), R)


            if y_f_enkf_tmp[2, i] <= 0 :
                print(f"particule i:{i} at k:{k} on land")
                x_f_enkf_tmp[:, i] = x_a_enkf_tmp[:, i]
                y_f_enkf_tmp[:, i] = h(x_f_enkf_tmp[:, i], pv_a[k-1] if use_pv else None, x_a_enkf_tmp[:,i] if use_speed else None, t_k=t_eval[k], t_km1=t_eval[k-1])

        x_f_enkf_full[:, :, k] = x_f_enkf_tmp        
        P_f_enkf_tmp = np.cov(x_f_enkf_tmp)




        valid_obs = np.isfinite(y[:, k])
        if np.any(valid_obs):
            y_f_enkf_tmp_valid = y_f_enkf_tmp[valid_obs, :]
            R_valid = R[np.ix_(valid_obs, valid_obs)]

            x_f_bar = x_f_enkf_tmp.mean(axis=1, keepdims=True)
            y_f_bar = y_f_enkf_tmp_valid.mean(axis=1, keepdims=True)
        
            diff_x = x_f_enkf_tmp - x_f_bar
            diff_y = y_f_enkf_tmp_valid - y_f_bar
        
            Pxy = diff_x @ diff_y.T / (Ne_enkf - 1)
            Pyy = diff_y @ diff_y.T / (Ne_enkf - 1) #+ R_valid  #if commented -> R1
        
            K = Pxy @ np.linalg.inv(Pyy)
        
            y_valid = y[valid_obs, k]

            # update ensemble
            for i in range(Ne_enkf):
                x_a_enkf_tmp[:, i] = x_f_enkf_tmp[:, i] + K @ (y_valid[:] - y_f_enkf_tmp_valid[:, i])

            P_a_enkf_tmp = np.cov(x_a_enkf_tmp)

        else:
            x_a_enkf_tmp = x_f_enkf_tmp.copy()
            P_a_enkf_tmp = P_f_enkf_tmp.copy()
    

    
        # store results
        x_f_enkf[:, k] = np.nanmean(x_f_enkf_tmp, axis=1)
        P_f_enkf[:, :, k] = P_f_enkf_tmp
        x_a_enkf[:, k] = np.nanmean(x_a_enkf_tmp, axis=1)
        P_a_enkf[:, :, k] = P_a_enkf_tmp
        x_a_enkf_full[:, :, k] = x_a_enkf_tmp

        if use_pv :
            pv_f[k] = pv_xy(x_f_enkf[0, k],x_f_enkf[1, k])
            pv_a[k] = pv_xy(x_a_enkf[0, k],x_a_enkf[1, k])


        if use_speed : 
            speed_a_enkf[k] = speed_xy(x_a_enkf[0, k],x_a_enkf[1, k], x_a_enkf[0, k-1],x_a_enkf[1, k-1], t_eval[k], t_eval[k-1])

        if np.any(np.isnan(x_a_enkf[:,k])):
            raise ValueError("NaN analyse")




    ### Ensemble Kalman smoother (EnKS)

        x_s_enkf = np.zeros((n,nb)) 
        P_s_enkf = np.zeros((n,n,nb))
        x_s_enkf_full = np.zeros((n, Ne_enkf, nb))
    
        x_s_enkf_full[:, :, -1] = x_a_enkf_full[:, :, -1]
    
        # statistiques au dernier pas
        x_s_enkf[:, -1] = np.mean(x_s_enkf_full[:, :, -1], axis=1)
        P_s_enkf[:, :, -1] = np.cov(x_s_enkf_full[:, :, -1])
    
        for k in range(nb-2, -1, -1):
            if k%10 == 0 : 
                print(f"{k}/{nb}")
            
            diff_a = x_a_enkf_full[:, :, k] - np.mean(x_a_enkf_full[:, :, k], axis=1, keepdims=True)
            diff_f = x_f_enkf_full[:, :, k+1] - np.mean(x_f_enkf_full[:, :, k+1], axis=1, keepdims=True)
    
            Paf = diff_a @ diff_f.T / (Ne_enkf - 1)
            Pf = np.cov(x_f_enkf_full[:, :, k+1])
    
            Ks = Paf @ np.linalg.inv(Pf)
    
            for i in range(Ne_enkf):
                x_s_enkf_full[:, i, k] = x_a_enkf_full[:, i, k] + Ks @ (x_s_enkf_full[:, i, k+1] - x_f_enkf_full[:, i, k+1])
    
            x_s_enkf[:, k] = np.mean(x_s_enkf_full[:, :, k], axis=1)
            P_s_enkf[:, :, k] = np.cov(x_s_enkf_full[:, :, k])

            if use_pv :
                pv_s[k] = pv_xy(x_s_enkf[0, k],x_s_enkf[1, k])



        x_enkf, y_enkf = x_s_enkf
        x_enkf_full, y_enkf_full = x_s_enkf_full
        P_enkf = P_s_enkf

    


    lon_enkf, lat_enkf = xy2lonlat(x_enkf, y_enkf)
    lon_enkf_full, lat_enkf_full = xy2lonlat(x_enkf_full, y_enkf_full)


## 7. Particle filter and FFBS

In [ ]:
if use_ffbs:
    ### Particle filter

    # Particle filter initialization
    x_f_pf = np.zeros((n, nb))
    P_f_pf = np.zeros((n, n, nb))
    x_f_pf_full = np.zeros((n, Ne_ffbs, nb))
    weights_f_full = np.zeros((Ne_ffbs, nb))

    x_pf_tmp = np.zeros((n, Ne_ffbs))
    y_pf_tmp = np.zeros((p, Ne_ffbs))

    x_pf_tmp = np.random.multivariate_normal(X_0, P_0, Ne_ffbs).T
    weights_tmp = np.ones(Ne_ffbs)/Ne_ffbs
    logw_tmp = np.log(weights_tmp)

    x_f_pf[:,0] = np.sum(x_pf_tmp*weights_tmp,axis=1)
    x_f_pf_full[:,:,0] = x_pf_tmp[:,:]
    weights_f_full[:, 0] = weights_tmp 
    dx0 = x_pf_tmp - x_f_pf[:,0,None]
    P_f_pf[:,:,0] = (dx0 * weights_tmp[None,:]) @ dx0.T

    if use_pv:
        pv0 = pv_xy(X_0[0], X_0[1])
        pv_f_pf = np.zeros(nb)
        pv_f_pf[0] = pv0
    
    if use_speed : 
        speed_f_pf = np.zeros((nb))
        speed_f_pf[0] = np.nan
        speed_f_pf_full = np.zeros((Ne_ffbs, nb))
        speed_f_pf_full[:,0] = np.nan


    for k in range(1,nb): # forward in time
        if k%10 == 0 : 
            print(f"{k}/{nb}")
        x_pf_past = x_pf_tmp.copy()
        dt = t_eval[k] - t_eval[k-1]
        time = times[k]
        Q = np.diag([(std_current*dt)**2, (std_current*dt)**2])

        z = parking_depth[k]
        Q_batch = np.random.multivariate_normal(np.zeros(n), Q, size=Ne_ffbs).T  # (n, Ne)


        if np.isfinite(y[0, k]) and np.isfinite(y[1, k]): # If a valid GNSS position is available, sample particles around it
            gps_position = y[:2, k]
            R_gps = np.diag([std_gps**2, std_gps**2])
            x_pf_tmp = (gps_position[:, None] + np.random.multivariate_normal(mean=np.zeros(2), cov=R_gps, size=Ne_ffbs).T)
            weights_tmp[:] = 1.0 / Ne_ffbs
            logw_tmp[:] = np.log(1.0 / Ne_ffbs)

        else :
            x_pf_tmp = m_batch(x_pf_past, t_eval[k-1], t_eval[k], times[k], times[k-1], z)  + Q_batch

            for i in range(Ne_ffbs):
                y_pf_tmp[:,i] = h(x_pf_tmp[:, i], pv_f_pf[k-1] if use_pv else None, x_pf_past[:,i] if use_speed else None, t_eval[k], t_eval[k-1])
                if y_pf_tmp[2,i] <= 0. :
                        print(f"particule i:{i} at k:{k} on land")
                        logw_tmp[i] = -np.inf
    
        
            valid_obs = np.isfinite(y[:, k])
            if np.any(valid_obs):
                y_pf_tmp_valid = y_pf_tmp[valid_obs, :]
                R_valid = R[np.ix_(valid_obs, valid_obs)]
                y_valid = y[valid_obs, k]
            
                for i in range(Ne_ffbs):
                    if logw_tmp[i] == -np.inf :
                        continue
                    else :  
                        innovation = y_valid - y_pf_tmp_valid[:,i]
                        logw_tmp[i] += -0.5 * innovation.T @ np.linalg.inv(R_valid) @ innovation
                logw_tmp -= np.max(logw_tmp)
                weights_tmp = np.exp(logw_tmp)
                weights_tmp += 1e-300           # to avoid null weights
                weights_tmp /= np.sum(weights_tmp)  # weight normalization        
                logw_tmp = np.log(weights_tmp) 
            
        # Weighted position estimate
        x_f_pf[:,k] = np.sum(x_pf_tmp*weights_tmp,axis=1)
        dx = x_pf_tmp - x_f_pf[:,k,None]
        P_f_pf[:,:,k] = (dx * weights_tmp[None,:]) @ dx.T
        x_f_pf_full[:,:,k] = x_pf_tmp[:,:]
        weights_f_full[:, k] = weights_tmp.copy()
    
        if use_pv:
            pv_f_pf[k] = pv_xy(x_f_pf[0, k], x_f_pf[1, k])

        if use_speed : 
            speed_f_pf[k] = speed_xy(x_f_pf[0, k],x_f_pf[1, k], x_f_pf[0, k-1],x_f_pf[1, k-1], t_eval[k], t_eval[k-1])
            speed_f_pf_full[:, k] = speed_xy(x_pf_tmp[0, :], x_pf_tmp[1, :], x_pf_past[0,:],x_pf_past[1,:], t_eval[k], t_eval[k-1])


        # particle resampling
        # Effective sample size
        Neff = 1.0 / np.sum(weights_tmp**2)
        if Neff < Ne_ffbs/2:
            cdf = np.cumsum(weights_tmp)
            start = np.random.uniform(0, 1/Ne_ffbs)
            positions = start + np.arange(Ne_ffbs)/Ne_ffbs
            indices = np.searchsorted(cdf, positions)
            # New equally weighted particles
            x_pf_tmp = x_pf_tmp[:, indices]
            weights_tmp[:] = 1.0/Ne_ffbs
            logw_tmp[:] = np.log(1.0/Ne_ffbs)


    if use_ffbs:
        x_s_pf = np.zeros((n, nb))
        weights_s_full = np.zeros((Ne_ffbs, nb))
        P_s_pf = np.zeros((n,n,nb))
    
        weights_s_full[:, -1] = weights_f_full[:, -1]
        x_s_pf[:, -1] = np.sum(x_f_pf_full[:, :, -1] * weights_s_full[:, -1], axis=1)
        dx_s = x_f_pf_full[:, :, -1] - x_s_pf[:, -1, None]
        P_s_pf[:, :, -1] = (dx_s * weights_s_full[None, :, -1]) @ dx_s.T

        if use_pv : 
            pv_s_pf = np.zeros((nb))
            pv_s_pf[-1] = pv_xy(x_s_pf[0, -1], x_s_pf[1, -1])

        for k in range(nb-2, -1, -1):
            if k % 10 == 0:
                print(f"FFBS {k}/{nb}")

            dt = t_eval[k+1] - t_eval[k]
            Qdiag = np.array([(std_current*dt)**2, (std_current*dt)**2])
            z = parking_depth[k+1]

            # Deterministic transition mean for each particle at time k
            means_pred = m_batch(x_f_pf_full[:, :, k], t_eval[k], t_eval[k+1], times[k+1], times[k], z)  # (n, Ne)

            # Unnormalized transition density for all particle pairs
            P_trans = np.zeros((Ne_ffbs, Ne_ffbs))
            for i in range(Ne_ffbs):
                diff = x_f_pf_full[:, :, k+1] - means_pred[:, i:i+1]  # (n, Ne)
                quad = np.sum((diff**2) / Qdiag[:, None], axis=0)   # (Ne,)
                P_trans[i, :] = np.exp(-0.5 * quad)

            # Backward recursion of smoothed particle weights
            denom = weights_f_full[:, k] @ P_trans          # (Ne,) : sum_l w_f(l,k) p(x_{k+1}(j)|x_k(l))
            denom = np.maximum(denom, 1e-300)
            ratio = P_trans / denom[None, :]               # ratio[i,j]

            weights_s_full[:, k] = weights_f_full[:, k] * (ratio @ weights_s_full[:, k+1])
            weights_s_full[:, k] /= np.sum(weights_s_full[:, k])

            x_s_pf[:, k] = np.sum(x_f_pf_full[:, :, k] * weights_s_full[:, k], axis=1)

            dx_s = x_f_pf_full[:, :, k] - x_s_pf[:, k, None]
            P_s_pf[:, :, k] = (dx_s * weights_s_full[None, :, k]) @ dx_s.T

            if use_pv:
                pv_s_pf[k] = pv_xy(x_s_pf[0, k], x_s_pf[1, k])



    if not use_ffbs : 
        x_pf, y_pf = x_f_pf
        P_pf = P_f_pf

    if use_ffbs : 
        x_pf, y_pf = x_s_pf
        P_pf = P_s_pf

    x_pf_full, y_pf_full = x_f_pf_full

    lon_pf, lat_pf = xy2lonlat(x_pf, y_pf)
    lon_pf_full, lat_pf_full = xy2lonlat(x_pf_full, y_pf_full)

## 8. Reconstruction preparation

In [ ]:
# Dictionary of selected reconstruction methods.
# Each method contains the reconstructed position and its covariance.

reconstructions = {}

if use_enks:
    reconstructions["EnKS"] = {
        "x": x_s_enkf,
        "y": y_s_enkf,
        "P": P_s_enkf,
        "lon": xy2lonlat(x_s_enkf[0], x_s_enkf[1])[0],
        "lat": xy2lonlat(x_s_enkf[0], x_s_enkf[1])[1],
        "color": color_enks,
    }

if use_ffbs:
    reconstructions["FFBS"] = {
        "x": x_s_pf,
        "y": y_s_pf,
        "P": P_s_pf,
        "lon": xy2lonlat(x_s_pf[0], x_s_pf[1])[0],
        "lat": xy2lonlat(x_s_pf[0], x_s_pf[1])[1],
        "color": color_ffbs,
    }

def semi_major_axis_95(P):
    """95% confidence ellipse semi-major axis in meters."""
    values = np.zeros(P.shape[-1])
    for k in range(P.shape[-1]):
        cov = 0.5 * (P[:, :, k] + P[:, :, k].T)
        if np.all(np.isfinite(cov)):
            eigvals = np.linalg.eigvalsh(cov)
            values[k] = np.sqrt(chi2_95 * max(eigvals.max(), 0.0))
        else:
            values[k] = np.nan
    return values

def ellipse_lonlat(x, y, P, n_points=80):
    """Geographic coordinates of a 95% confidence ellipse."""
    cov = 0.5 * (P + P.T)
    if not np.all(np.isfinite(cov)):
        return None, None

    eigvals, eigvecs = np.linalg.eigh(cov)
    eigvals = np.maximum(eigvals, 0)
    order = np.argsort(eigvals)[::-1]
    eigvals = eigvals[order]
    eigvecs = eigvecs[:, order]

    axes = np.sqrt(chi2_95 * eigvals)
    theta = np.linspace(0, 2*np.pi, n_points)
    ellipse = eigvecs @ np.diag(axes) @ np.vstack([np.cos(theta), np.sin(theta)])
    ellipse[0] += x
    ellipse[1] += y

    return xy2lonlat(ellipse[0], ellipse[1])

# Check results
for name, data in reconstructions.items():
    data["a95_km"] = semi_major_axis_95(data["P"]) / 1000
    print(f"{name}: reconstruction available ({len(data['lon'])} positions)")


## 9. Reconstruction maps

In [ ]:
def plot_reconstruction(method_names, filename=None):
    """
    Plot one map for each selected method.
    If both methods are selected, add a third map comparing them.
    The recorded trajectory is shown as the pre-reconstruction reference.
    The ellipses represent the 95% confidence region.
    """
    n_methods = len(method_names)
    n_maps = n_methods + (1 if n_methods == 2 else 0)

    fig, axes = plt.subplots(
        1, n_maps,
        figsize=(8 * n_maps, 8),
        subplot_kw={"projection": ccrs.NorthPolarStereo()}
    )
    axes = np.atleast_1d(axes)

    plot_slice = slice(plot_start_cycle, plot_end_cycle)

    recorded_lon = cycle_lon[plot_slice]
    recorded_lat = cycle_lat[plot_slice]
    gnss_lon = lons[plot_slice]
    gnss_lat = lats[plot_slice]

    # Common map background
    bathy_sub = bathy_ds.sel(
        longitude=slice(map_extent[0] - 20, map_extent[1] + 20),
        latitude=slice(map_extent[2], map_extent[3])
    )
    Lon_bathy, Lat_bathy = np.meshgrid(
        bathy_sub.longitude.values,
        bathy_sub.latitude.values
    )
    bathy_values = bathy_sub.bathymetry.values

    vmin = np.nanquantile(bathy_values, 0.05)
    vmax = np.nanquantile(bathy_values, 0.95)

    def setup_ax(ax, title):
        ax.set_extent(map_extent, crs=ccrs.PlateCarree())
        ax.pcolormesh(
            Lon_bathy, Lat_bathy, bathy_values,
            transform=ccrs.PlateCarree(),
            shading="auto", cmap="Blues", alpha=0.7,
            vmin=vmin, vmax=vmax, zorder=0
        )
        ax.add_feature(cfeature.LAND, zorder=1, edgecolor="black", facecolor="lightgray")
        ax.gridlines(draw_labels=False, linewidth=0.5, alpha=0.3)
        ax.set_title(title, fontsize=15)

        # Recorded trajectory used as the pre-reconstruction reference
        ax.plot(
            recorded_lon, recorded_lat,
            transform=ccrs.PlateCarree(),
            color=color_recorded, linewidth=2,
            label="Recorded trajectory", zorder=4
        )

        # Valid GNSS observations
        ax.plot(
            gnss_lon, gnss_lat,
            transform=ccrs.PlateCarree(),
            color=color_gnss, linewidth=0,
            marker="o", markersize=3,
            label="GNSS observations", zorder=6
        )

    # One map per selected method
    for j, method in enumerate(method_names):
        ax = axes[j]
        data = reconstructions[method]
        setup_ax(ax, method)

        ax.plot(
            data["lon"][plot_slice], data["lat"][plot_slice],
            transform=ccrs.PlateCarree(),
            color=data["color"], linewidth=2,
            marker="o", markersize=3,
            label=f"{method} reconstruction", zorder=5
        )

        k_start = plot_start_cycle
        k_end = nb if plot_end_cycle is None else min(plot_end_cycle, nb)
        for k in range(k_start, k_end, ellipse_step):
            lon_ell, lat_ell = ellipse_lonlat(
                data["x"][k], data["y"][k], data["P"][:, :, k]
            )
            if lon_ell is not None:
                ax.plot(
                    lon_ell, lat_ell,
                    transform=ccrs.PlateCarree(),
                    color=data["color"], linewidth=0.8,
                    alpha=0.65, zorder=5
                )

        ax.legend(loc="best", fontsize=9)

    # Combined map when both methods are selected
    if n_methods == 2:
        ax = axes[-1]
        setup_ax(ax, "EnKS + FFBS")

        k_start = plot_start_cycle
        k_end = nb if plot_end_cycle is None else min(plot_end_cycle, nb)

        for method in method_names:
            data = reconstructions[method]
            ax.plot(
                data["lon"][plot_slice], data["lat"][plot_slice],
                transform=ccrs.PlateCarree(),
                color=data["color"], linewidth=2,
                marker="o", markersize=3,
                label=f"{method} reconstruction", zorder=5
            )

            for k in range(k_start, k_end, ellipse_step):
                lon_ell, lat_ell = ellipse_lonlat(
                    data["x"][k], data["y"][k], data["P"][:, :, k]
                )
                if lon_ell is not None:
                    ax.plot(
                        lon_ell, lat_ell,
                        transform=ccrs.PlateCarree(),
                        color=data["color"], linewidth=0.8,
                        alpha=0.65, zorder=5
                    )

        ax.legend(loc="best", fontsize=9)

    fig.suptitle(
        f"Trajectory reconstruction — WMO {float_name}",
        fontsize=18
    )
    fig.tight_layout()

    if filename is not None:
        fig.savefig(filename, dpi=300, bbox_inches="tight")

    plt.show()
    return fig

method_names = list(reconstructions.keys())

plot_reconstruction(
    method_names,
    figure_dir / f"reconstruction_{float_name}.png"
)


## 10. Position uncertainty over time

In [ ]:
fig, ax = plt.subplots(figsize=figsize_uncertainty)

for method, data in reconstructions.items():
    ax.plot(
        times,
        data["a95_km"],
        color=data["color"],
        linewidth=2,
        label=f"{method} — demi-grand axe 95 %"
    )

ax.set_xlabel("Date")
ax.set_ylabel("Position uncertainty — 95% semi-major axis [km]")
ax.set_title(f"Position uncertainty over time — WMO {float_name}")
ax.grid(alpha=0.3)
ax.legend()
fig.tight_layout()

fig.savefig(
    figure_dir / f"uncertainty_{float_name}.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()


## 11. Save reconstructed trajectories

In [ ]:
# One CSV file containing all selected methods.
# Projected coordinates and uncertainty are stored in addition to longitude/latitude.

df_output = pd.DataFrame({
    "time": pd.to_datetime(times),
    "recorded_lon": cycle_lon,
    "recorded_lat": cycle_lat,
    "gnss_lon": lons,
    "gnss_lat": lats,
    "gnss_valid": valid,
})

for method, data in reconstructions.items():
    prefix = method.lower()

    df_output[f"{prefix}_lon"] = data["lon"]
    df_output[f"{prefix}_lat"] = data["lat"]
    df_output[f"{prefix}_x_m"] = data["x"][0]
    df_output[f"{prefix}_y_m"] = data["y"][0]
    df_output[f"{prefix}_sigma_x_m"] = np.sqrt(np.maximum(data["P"][0, 0, :], 0))
    df_output[f"{prefix}_sigma_y_m"] = np.sqrt(np.maximum(data["P"][1, 1, :], 0))
    df_output[f"{prefix}_cov_xy_m2"] = data["P"][0, 1, :]
    df_output[f"{prefix}_uncertainty_95_major_km"] = data["a95_km"]

csv_file = csv_dir / f"reconstructed_trajectory_{float_name}.csv"
df_output.to_csv(csv_file, index=False)

print(f"Trajectory saved to: {csv_file}")
display(df_output.head())
